# KWGT Harvest and Extract from GitHub

This notebook searches GitHub for KWGT files, downloads them, and extracts preset.json for analysis.

## Features
- Search GitHub repositories for .kwgt files
- Download .kwgt files using optional GH_TOKEN
- Extract preset.json only
- Mine internal_type registry
- Export results as ZIP

In [ ]:
# Install dependencies
!pip install -q requests PyGithub

In [ ]:
import os
import json
import zipfile
import requests
from pathlib import Path
from collections import defaultdict, Counter
from github import Github
from io import BytesIO
import base64
from getpass import getpass

# Configuration
OUTPUT_DIR = Path('/content/kwgt_github_output')
DOWNLOADS_DIR = OUTPUT_DIR / 'downloads'
EXTRACTED_DIR = OUTPUT_DIR / 'extracted_presets'
REGISTRY_FILE = OUTPUT_DIR / 'internal_type_registry.json'
ANALYSIS_FILE = OUTPUT_DIR / 'schema_analysis.json'
METADATA_FILE = OUTPUT_DIR / 'github_metadata.json'

# Create output directories
OUTPUT_DIR.mkdir(exist_ok=True)
DOWNLOADS_DIR.mkdir(exist_ok=True)
EXTRACTED_DIR.mkdir(exist_ok=True)

print("Output directories created")

In [ ]:
# GitHub token (optional but recommended to avoid rate limits)
# Set GH_TOKEN environment variable or enter when prompted

GH_TOKEN = os.environ.get('GH_TOKEN', '')

if not GH_TOKEN:
    print("⚠️  No GH_TOKEN found in environment")
    print("   You can continue without a token, but you may hit rate limits.")
    print("   To use a token, enter it below (leave empty to skip):")
    GH_TOKEN = getpass("GitHub Token (optional): ")

# Initialize GitHub client
if GH_TOKEN:
    g = Github(GH_TOKEN)
    print("✅ GitHub client initialized with token")
    user = g.get_user()
    rate_limit = g.get_rate_limit()
    print(f"   Authenticated as: {user.login}")
    print(f"   Rate limit: {rate_limit.core.remaining}/{rate_limit.core.limit}")
else:
    g = Github()
    print("⚠️  Using unauthenticated GitHub client (limited to 60 requests/hour)")

In [ ]:
# Search parameters
SEARCH_QUERY = 'extension:kwgt'
MAX_RESULTS = 50  # Limit to avoid excessive downloads

print(f"Searching GitHub for .kwgt files...")
print(f"Query: {SEARCH_QUERY}")
print(f"Max results: {MAX_RESULTS}")
print("="*60)

In [ ]:
def download_file(url, token=None):
    """
    Download a file from GitHub.
    
    Args:
        url: URL to download from
        token: Optional GitHub token for authentication
    
    Returns:
        BytesIO object with file content or None if failed
    """
    headers = {}
    if token:
        headers['Authorization'] = f'token {token}'
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        return BytesIO(response.content)
    except Exception as e:
        print(f"   ❌ Download failed: {e}")
        return None

def extract_preset_from_bytes(kwgt_bytes, output_path):
    """
    Extract preset.json from .kwgt file bytes.
    
    Args:
        kwgt_bytes: BytesIO object with .kwgt content
        output_path: Path to save extracted preset.json
    
    Returns:
        True if successful, False otherwise
    """
    try:
        with zipfile.ZipFile(kwgt_bytes, 'r') as zip_ref:
            if 'preset.json' not in zip_ref.namelist():
                return False
            
            preset_data = zip_ref.read('preset.json')
            with open(output_path, 'wb') as f:
                f.write(preset_data)
            
            return True
    except Exception as e:
        print(f"   ⚠️  Extraction error: {e}")
        return False

def collect_internal_types(obj, types_dict=None, path="root"):
    """
    Recursively collect all internal_type values.
    """
    if types_dict is None:
        types_dict = defaultdict(list)
    
    if isinstance(obj, dict):
        if 'internal_type' in obj:
            internal_type = obj['internal_type']
            types_dict[internal_type].append({
                'path': path,
                'keys': list(obj.keys())[:20]
            })
        
        for key, value in obj.items():
            collect_internal_types(value, types_dict, f"{path}.{key}")
    
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            collect_internal_types(item, types_dict, f"{path}[{i}]")
    
    return types_dict

def analyze_schema(preset_data):
    """
    Analyze KBM schema structure.
    """
    analysis = {
        'has_root_layer': 'root_layer' in preset_data,
        'has_globals': 'globals' in preset_data,
        'has_items': 'items' in preset_data,
        'globals_count': len(preset_data.get('globals', [])),
        'items_count': len(preset_data.get('items', [])),
        'root_type': preset_data.get('root_layer', {}).get('internal_type', 'N/A'),
    }
    
    return analysis

In [ ]:
# Search for .kwgt files on GitHub
print("Searching...\n")

try:
    search_results = g.search_code(query=SEARCH_QUERY, order='desc')
    
    # Collect results
    kwgt_files = []
    for result in search_results[:MAX_RESULTS]:
        kwgt_files.append({
            'name': result.name,
            'path': result.path,
            'repo': result.repository.full_name,
            'download_url': result.download_url,
            'html_url': result.html_url,
            'size': result.size,
        })
    
    print(f"✅ Found {len(kwgt_files)} .kwgt files")
    
    # Show first few results
    print("\nFirst 5 results:")
    for i, file in enumerate(kwgt_files[:5], 1):
        print(f"  {i}. {file['name']}")
        print(f"     Repo: {file['repo']}")
        print(f"     Size: {file['size']/1024:.1f} KB")

except Exception as e:
    print(f"❌ Search failed: {e}")
    kwgt_files = []

In [ ]:
# Download and process files
print("\n" + "="*60)
print("Downloading and extracting...\n")

all_internal_types = defaultdict(list)
schema_analyses = []
metadata = []

success_count = 0
fail_count = 0

for i, file_info in enumerate(kwgt_files, 1):
    print(f"[{i}/{len(kwgt_files)}] {file_info['name']}")
    print(f"  Repo: {file_info['repo']}")
    
    try:
        # Download file
        kwgt_bytes = download_file(file_info['download_url'], GH_TOKEN)
        if not kwgt_bytes:
            fail_count += 1
            continue
        
        # Save original .kwgt file
        kwgt_path = DOWNLOADS_DIR / file_info['name']
        with open(kwgt_path, 'wb') as f:
            f.write(kwgt_bytes.getvalue())
        
        # Reset BytesIO position
        kwgt_bytes.seek(0)
        
        # Extract preset.json
        preset_path = EXTRACTED_DIR / f"{kwgt_path.stem}_preset.json"
        if not extract_preset_from_bytes(kwgt_bytes, preset_path):
            print("  ⚠️  No preset.json found")
            fail_count += 1
            continue
        
        # Load and analyze preset
        with open(preset_path, 'r', encoding='utf-8') as f:
            preset_data = json.load(f)
        
        # Collect internal types
        types_dict = collect_internal_types(preset_data)
        for itype, contexts in types_dict.items():
            all_internal_types[itype].extend(contexts)
        
        # Analyze schema
        analysis = analyze_schema(preset_data)
        analysis['filename'] = file_info['name']
        analysis['repo'] = file_info['repo']
        schema_analyses.append(analysis)
        
        # Save metadata
        metadata.append(file_info)
        
        print(f"  ✅ Success - {len(types_dict)} types, {analysis['items_count']} items")
        success_count += 1
    
    except Exception as e:
        print(f"  ❌ Error: {e}")
        fail_count += 1
    
    # Rate limit check every 10 files
    if i % 10 == 0 and GH_TOKEN:
        rate_limit = g.get_rate_limit()
        print(f"  ℹ️  Rate limit: {rate_limit.core.remaining}/{rate_limit.core.limit}")

print("\n" + "="*60)
print(f"\n📊 Summary:")
print(f"   Total files: {len(kwgt_files)}")
print(f"   Successful: {success_count}")
print(f"   Failed: {fail_count}")
print(f"   Unique internal_types: {len(all_internal_types)}")

In [ ]:
# Generate internal_type registry
registry = {}
for itype, contexts in all_internal_types.items():
    all_keys = set()
    for ctx in contexts:
        all_keys.update(ctx['keys'])
    
    registry[itype] = {
        'count': len(contexts),
        'common_keys': sorted(list(all_keys)),
        'example_paths': list(set(ctx['path'] for ctx in contexts[:5]))
    }

# Save registry
with open(REGISTRY_FILE, 'w', encoding='utf-8') as f:
    json.dump(registry, f, indent=2)

print(f"\n📋 Internal Type Registry:")
print(f"   Saved to: {REGISTRY_FILE}")
print(f"   Types: {len(registry)}")
print("\nTop 10 most common types:")
sorted_types = sorted(registry.items(), key=lambda x: x[1]['count'], reverse=True)[:10]
for itype, data in sorted_types:
    print(f"   - {itype}: {data['count']} occurrences")

In [ ]:
# Save schema analysis
with open(ANALYSIS_FILE, 'w', encoding='utf-8') as f:
    json.dump(schema_analyses, f, indent=2)

# Save metadata
with open(METADATA_FILE, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print(f"\n📊 Analysis saved:")
print(f"   Schema analysis: {ANALYSIS_FILE}")
print(f"   GitHub metadata: {METADATA_FILE}")
print(f"   Files analyzed: {len(schema_analyses)}")

# Summary statistics
if schema_analyses:
    root_types = Counter(a['root_type'] for a in schema_analyses)
    avg_globals = sum(a['globals_count'] for a in schema_analyses) / len(schema_analyses)
    avg_items = sum(a['items_count'] for a in schema_analyses) / len(schema_analyses)
    
    print(f"\nStatistics:")
    print(f"   Root types: {dict(root_types)}")
    print(f"   Avg globals per widget: {avg_globals:.1f}")
    print(f"   Avg items per widget: {avg_items:.1f}")

In [ ]:
# Create ZIP archive of all outputs
zip_output = Path('/content') / 'kwgt_github_harvest.zip'

with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add analysis files
    zipf.write(REGISTRY_FILE, REGISTRY_FILE.name)
    zipf.write(ANALYSIS_FILE, ANALYSIS_FILE.name)
    zipf.write(METADATA_FILE, METADATA_FILE.name)
    
    # Add extracted presets
    for preset_file in EXTRACTED_DIR.glob('*.json'):
        zipf.write(preset_file, f"extracted_presets/{preset_file.name}")
    
    # Add downloaded .kwgt files
    for kwgt_file in DOWNLOADS_DIR.glob('*.kwgt'):
        zipf.write(kwgt_file, f"downloads/{kwgt_file.name}")

print(f"\n📦 Results packaged:")
print(f"   ZIP file: {zip_output}")
print(f"   Size: {zip_output.stat().st_size / 1024:.1f} KB")
print(f"\n✅ GitHub harvest complete! Download the ZIP file from the Files panel.")

## Usage Instructions

1. (Optional) Set `GH_TOKEN` environment variable or enter when prompted
2. Run all cells in order
3. Wait for search and download to complete
4. Download the `kwgt_github_harvest.zip` file from the Files panel

## Output Structure

```
kwgt_github_harvest.zip
├── internal_type_registry.json  # Registry of all internal_type values
├── schema_analysis.json         # Schema analysis for each widget
├── github_metadata.json         # GitHub source information
├── extracted_presets/           # All extracted preset.json files
│   ├── widget1_preset.json
│   └── ...
└── downloads/                   # Original .kwgt files
    ├── widget1.kwgt
    └── ...
```

## Rate Limits

- **Without token**: 60 requests/hour
- **With token**: 5000 requests/hour

Use a GitHub token to avoid rate limit issues.